# Security Services

Identity and governance from notebook 02 give you the *who* and *what allowed*. This notebook is about the rest of the security stack — the services that hold secrets, prove workloads to other services without secrets, watch the platform for misconfiguration, hunt for active attacks, and absorb the volumetric and application-layer assaults that hit any internet-facing system.

The five products to know: **Key Vault** holds keys, secrets, and certificates. **Managed Identities** make workloads provable. **Microsoft Defender for Cloud** scores your security posture and protects workloads. **Microsoft Sentinel** is the SIEM/SOAR. **DDoS Protection** and **Web Application Firewalls** sit at the network and application edges respectively. Together they turn Azure from a platform you have to bolt security onto into one where security composes with the rest of the stack.

## Azure Key Vault

**Azure Key Vault** stores three classes of cryptographic material:

- **Keys** — asymmetric (RSA, EC) or symmetric (AES, on Managed HSM). Used for encrypt/decrypt, sign/verify, wrap/unwrap. Customer-managed-key (CMK) scenarios (storage, disk, SQL) point at a key in Key Vault.
- **Secrets** — opaque strings up to 25 KB. Connection strings, API tokens, anything you cannot eliminate with a managed identity.
- **Certificates** — x509 certs with their private key. Auto-renew via integration with public CAs (DigiCert, GlobalSign) or via the **App Service managed cert** path.

Two SKUs of Key Vault itself:

- **Standard / Premium vault** — multi-tenant software-backed (Standard) or HSM-backed (Premium) key storage. Premium uses FIPS 140-2 Level 2 validated HSMs shared across tenants.
- **Managed HSM** — single-tenant, dedicated HSM cluster (FIPS 140-2 Level 3). Required for regulated workloads that mandate exclusive HSM use. Expensive (per-hour billing); reserve for genuine compliance need.

**Soft delete** and **purge protection** are the lifesaving features. Soft delete retains deleted vaults and objects for 7–90 days; purge protection blocks the *permanent* delete during the retention window. Microsoft now requires both on vaults that back encryption for managed services — without them, an accidental `az keyvault delete` could orphan production data.

**Access model**: Key Vault supports two modes for granting access — legacy **Access Policies** (the original 1:1 principal-to-permissions mapping) and **RBAC** (uses Azure RBAC like the rest of the platform). **RBAC is the current recommendation** for new vaults; Microsoft is steering everyone off Access Policies. Roles like `Key Vault Secrets User`, `Key Vault Crypto Officer`, `Key Vault Certificates Officer` keep the data-plane permissions separated from the management plane.

**Key rotation** for keys is automatic when you set a rotation policy on the key — Key Vault generates a new version at the configured interval; consumers using `https://<vault>/keys/<name>` (no version) pick up the new version on next call. For secrets, rotation requires a custom job (Function or Logic App) that generates the new value and updates the downstream service.

AWS comparison: Key Vault ≈ AWS Secrets Manager + AWS KMS combined; Managed HSM ≈ AWS CloudHSM.

## Managed Identities — secret-less auth

Notebook 02 introduced **managed identities** in the context of identity. They earn a deeper look here because *getting them right is the single highest-leverage move on credential hygiene*.

The two flavours, briefly:

- **System-assigned MI** — created with the resource, tied to its lifecycle. One-to-one.
- **User-assigned MI** — standalone resource, attachable to many resources, survives any one of them.

How code uses them: the Azure SDK's `DefaultAzureCredential` walks an ordered chain of auth sources — `EnvironmentCredential`, then `WorkloadIdentityCredential`, then `ManagedIdentityCredential`, then `AzureCliCredential`, etc. On an Azure VM with a managed identity attached, the SDK calls IMDS (`http://169.254.169.254/metadata/identity/...`) to get a token. Your code never sees a secret.

Three practical patterns:

- A **VM, App Service, Function, Container App, AKS pod** wants to read a secret from Key Vault → attach a managed identity, grant it `Key Vault Secrets User`, fetch secrets with the SDK.
- An **AKS workload** authenticates to a database → use **Azure AD Workload Identity** (federation between Kubernetes service accounts and Entra ID app registrations). No service-principal secrets in the cluster.
- **GitHub Actions / GitLab / external CI** deploys to Azure → use a **federated credential** on an Entra ID app registration. The pipeline's OIDC token is exchanged for an Entra ID token. No client secret in GitHub.

The end state worth aiming for: **zero secrets in pipelines, source control, or app config**. Everything that needs an Azure call uses an identity. Key Vault stores only the credentials that absolutely cannot be eliminated (third-party API keys, legacy stuff).

## Microsoft Defender for Cloud

**Microsoft Defender for Cloud (MDC)** is two products in one wrapper:

- **CSPM (Cloud Security Posture Management)** — free baseline that continuously assesses your Azure resources against the Microsoft Cloud Security Benchmark and produces a **secure score**. Findings are concrete ("this VM has no endpoint protection", "this storage account allows public blob access") with remediation steps. The free tier covers Azure; paid Defender CSPM extends to AWS, GCP, and on-prem via Azure Arc.
- **Workload Protection plans** — pay-per-resource agents that detect threats at runtime:
  - **Defender for Servers** (P1/P2) — EDR via Microsoft Defender for Endpoint, file integrity monitoring, just-in-time VM access.
  - **Defender for Containers** — image scanning in ACR, runtime detection on AKS, Kubernetes admission policies.
  - **Defender for Storage** — malware scanning on blob upload, anomalous access detection.
  - **Defender for SQL** — vulnerability assessment, anomalous-query detection.
  - **Defender for Key Vault**, **Defender for App Service**, **Defender for Cosmos DB**, etc.

The **regulatory compliance** dashboard maps your resources against ISO 27001, PCI DSS, NIST 800-53, HIPAA HITRUST, and other standards — useful when an auditor asks where you are.

Defender for Cloud is where security finds out what's wrong; Sentinel is where security responds. The two integrate — MDC alerts flow into Sentinel as incidents.

AWS comparison: Defender for Cloud ≈ Security Hub + GuardDuty + Inspector + Macie + Trusted Advisor combined into one console.

## Microsoft Sentinel

**Microsoft Sentinel** is the cloud-native SIEM (security information and event management) and SOAR (security orchestration and automated response). It runs on a Log Analytics workspace and is priced per GB ingested.

Four layers:

- **Data connectors** — ~150 built-in connectors pull logs from Azure (Activity Log, Defender for Cloud, Entra ID sign-ins, Storage, Key Vault), other clouds (AWS CloudTrail, GCP), Microsoft 365 (Defender XDR, Office), and third-party (Palo Alto, Cisco, Okta, syslog, CEF).
- **Analytics rules** — KQL queries that run on a schedule (e.g. every 5 minutes) and create **incidents** when they match. Microsoft ships hundreds of rule templates; you customise to your environment.
- **Incidents** — analyst workflow: triage, assign, investigate, close. Entities (users, IPs, hosts) link to related events automatically.
- **Playbooks** — Logic Apps that automate response — disable a user, isolate a host via Defender, post to Teams, open a ServiceNow ticket. Trigger automatically from rules or manually from incidents.

**UEBA** (User and Entity Behavior Analytics) baselines normal behaviour and flags deviations. **Threat intelligence** ingests indicators (IOCs) you upload or buy. **Notebooks** (Jupyter, with built-in connectors to the workspace) for ad-hoc hunting.

Sentinel is the right call when you have *several* Azure subscriptions, *or* multi-cloud, *or* a security operations centre. For a single small subscription, Defender for Cloud's alerts may be enough.

AWS comparison: Sentinel ≈ Security Hub + Detective + EventBridge + Athena queries, with the SIEM/SOAR experience genuinely more mature than the AWS equivalents.

## DDoS Protection

Every Azure region has **DDoS Protection Basic** built in for free — it absorbs the volumetric noise that hits the platform fabric. You don't see it; it just works.

**DDoS Protection Standard / IP Protection** is the paid tier that adds:

- **Adaptive tuning** of mitigation policies per public IP, based on traffic baselining.
- **Attack analytics** — flow logs, mitigation reports, post-attack analysis.
- **Cost protection** — Microsoft credits the egress and scale-out costs incurred during an attack.
- **24/7 access** to the DDoS rapid response team during active attacks.

Two SKUs in the Standard family:

- **DDoS Network Protection** — covers all public IPs in a VNet under one plan. Flat fee.
- **DDoS IP Protection** — pay per public IP, useful when you have a handful of high-value endpoints and don't need network-wide coverage.

Anything internet-facing in production — App Gateway, Front Door, Load Balancer with public IP — should be covered. Front Door has its own DDoS protection at the edge as part of the service, which is why Front Door + DDoS Standard on the regional public IPs is a common stacked posture.

AWS comparison: Basic ≈ AWS Shield Standard; Standard ≈ AWS Shield Advanced.

## Web Application Firewalls — positioning

Azure offers WAF on two products, and the choice is about *where* the WAF runs:

- **WAF on Application Gateway** — runs in your region, integrates with App Gateway listeners. Use when traffic is regional, or when you want WAF rules that can reference internal backend behaviour.
- **WAF on Azure Front Door (Premium)** — runs at the edge in 100+ POPs. Stops attacks before they ever reach your region. The default for internet-facing global apps.

Both share the same rule-set technology: **Microsoft Default Rule Set (DRS)** as the production default, plus **Bot Manager** rules for known bots, plus the **OWASP Core Rule Set** when you need standards alignment. Custom rules let you add your own logic — IP allow-lists, geo blocks, rate limits.

Two modes per policy: **Detection** (log only) and **Prevention** (block matching requests). Deploy in Detection first, watch the false-positive rate, then flip to Prevention.

Stacking Front Door WAF + App Gateway WAF is common: Front Door blocks the broad volumetric and OWASP-class attacks at the edge; App Gateway adds region-level rules that may reference backend behaviour. Each catches what the other can't.

AWS comparison: WAF on App Gateway / Front Door ≈ AWS WAF on ALB / CloudFront. The model is essentially the same.

In [ ]:
# A Key Vault + system-assigned managed identity flow.

RG=rg-sec-demo
KV=kv-foundations-$RANDOM
az group create -n $RG -l eastus

# 1. RBAC-mode vault with soft delete + purge protection.
az keyvault create -g $RG -n $KV \
  --enable-rbac-authorization true \
  --enable-soft-delete true \
  --retention-days 30 \
  --enable-purge-protection true

# 2. Store a secret.
az keyvault secret set --vault-name $KV --name "db-conn" --value "Server=..."

# 3. Create a Function App with system-assigned managed identity.
az functionapp create -g $RG -n func-secrets-demo \
  --storage-account stfdemo$RANDOM \
  --consumption-plan-location eastus \
  --runtime python --runtime-version 3.11 \
  --functions-version 4 --assign-identity

# 4. Grant the Function's MI permission to read secrets.
FUNC_MI=$(az functionapp identity show -g $RG -n func-secrets-demo --query principalId -o tsv)
az role assignment create \
  --assignee-object-id $FUNC_MI --assignee-principal-type ServicePrincipal \
  --role "Key Vault Secrets User" \
  --scope $(az keyvault show -n $KV --query id -o tsv)

# 5. Reference the secret from app settings as `@Microsoft.KeyVault(...)`.
az functionapp config appsettings set -g $RG -n func-secrets-demo \
  --settings "DB_CONN=@Microsoft.KeyVault(VaultName=$KV;SecretName=db-conn)"

## Putting it together

The security stack for a production workload, top down:

1. **Identity** — Entra ID with Conditional Access; admin roles via PIM; managed identities everywhere code calls Azure (notebook 02).
2. **Secrets** — Key Vault (RBAC mode, soft delete + purge protection) for the residual secrets that managed identities cannot eliminate. Auto-rotation policies on keys; rotation jobs for secrets.
3. **Workload protection** — Defender for Servers / Containers / SQL / Storage / Key Vault enabled on the resource types that hold customer data or face the internet.
4. **Posture** — Defender for Cloud's secure score reviewed weekly; regulatory compliance dashboard mapped to the standards the organisation cares about.
5. **Detect & respond** — Sentinel workspace ingesting Activity Log, Defender alerts, Entra ID sign-ins, and any third-party feeds; analytics rules tuned to the environment; high-severity playbooks automated (disable user, isolate host).
6. **Edge** — Front Door Premium WAF in Prevention; DDoS Network Protection on the regional VNet; App Gateway WAF stacked for region-level rules.

Get those six right and most of the breach surface area is closed off, and the surface area that remains is monitored, alerted on, and procedurally answered. Security in the cloud is not magic; it is the discipline of composing these primitives so that the easy thing and the secure thing are the same thing.